In [1]:
import numpy as np
from scipy.stats import randint, uniform
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
import joblib

# 1) Load MNIST
mnist = fetch_openml('mnist_784', as_frame=False)
X = mnist.data.astype(np.uint8)    # 👉 굳이 float32 + 정규화 안 해도 됨 (RF는 스케일에 둔감)
y = mnist.target.astype(np.uint8)


# X = X.astype(np.float32) / 255.0

# 2) 🔹 하이퍼파라미터 탐색용 서브셋 (예: 전체의 1/3만 사용)
subset_idx = np.arange(0, X.shape[0], 3)   # 0, 3, 6, ... → 약 23k 샘플
X_sub = X[subset_idx]
y_sub = y[subset_idx]

# 3) 기본 모델 (서치 단계에서는 나무 개수도 줄여서 가볍게)
rf_base = RandomForestClassifier(
    n_estimators=120,       # 🔹 서치 단계에서는 작게
    random_state=42,
    n_jobs=2,               # 🔹 메모리 터지는 것 방지 (문제 있으면 1로)
)

# 4) 탐색 범위 축소 (메모리/시간 고려해서 현실적으로)
param_distributions = {
    "max_depth": [None, 20, 30],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", 0.5],
    "bootstrap": [True],               # 🔹 False 제거 (무거움)
    "criterion": ["gini", "entropy"],
    # class_weight는 일단 고정 (필요하면 나중에 "balanced_subsample" 추가)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=20,              # 🔹 60 → 20으로 축소
    scoring="accuracy",
    cv=cv,
    n_jobs=2,               # 🔹 병렬 수 제한
    verbose=1,
    random_state=42,
    refit=True
)

print("=== RandomizedSearchCV (subset) 시작 ===")
search.fit(X_sub, y_sub)

print("\n=== RandomizedSearchCV 결과 (subset) ===")
print("Best CV Accuracy (subset): {:.4f}".format(search.best_score_))
print("Best Params:")
for k, v in search.best_params_.items():
    print(f"  - {k}: {v}")

best_params = search.best_params_


=== RandomizedSearchCV (subset) 시작 ===
Fitting 3 folds for each of 20 candidates, totalling 60 fits

=== RandomizedSearchCV 결과 (subset) ===
Best CV Accuracy (subset): 0.9497
Best Params:
  - bootstrap: True
  - criterion: entropy
  - max_depth: 20
  - max_features: log2
  - min_samples_leaf: 2
  - min_samples_split: 2


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# 1) 위에서 찾은 best params 그대로 사용
best_params = {
    "bootstrap": True,
    "criterion": "entropy",
    "max_depth": 20,
    "max_features": "log2",
    "min_samples_leaf": 2,
    "min_samples_split": 2,
}

# 2) 최종 모델 설정
best_params_for_final = best_params.copy()
best_params_for_final["n_estimators"] = 400   # 🔹 처음엔 300~400 정도 추천
best_params_for_final["n_jobs"] = 2          # 🔹 메모리 여유 없으면 1로 줄이기
best_params_for_final["random_state"] = 42

final_model = RandomForestClassifier(**best_params_for_final)

print("\n=== 전체 70,000개로 최종 모델 학습 시작 ===")
final_model.fit(X, y)   # 🔹 여기서 X, y는 전체 MNIST (정규화 안 한 uint8 버전이든, 너가 쓰는 그대로면 OK)

# 3) 훈련 데이터 기준 정확도 (참고용)
y_pred_all = final_model.predict(X)
train_acc = accuracy_score(y, y_pred_all)
print("전체 데이터 기준 예측 정확도(참고): {:.4f}".format(train_acc))

# 4) 모델 저장
joblib.dump(final_model, "rf_mnist_best_final.joblib")
print("\n모델 저장 완료: rf_mnist_best_final.joblib")



=== 전체 70,000개로 최종 모델 학습 시작 ===
전체 데이터 기준 예측 정확도(참고): 0.9988

모델 저장 완료: rf_mnist_best_final.joblib
